# Novellunar Scraper - Pick Me Up, Infinite Gacha

This notebook will guide you cell-by-cell to extract all 400 chapters from the website and save them locally as clean Markdown (`.md`) files.

## Step 1: Install Dependencies (If needed)
Uncomment and run the cell below if you do not have `requests` and `beautifulsoup4` installed in your environment.

In [ ]:
# !pip install requests beautifulsoup4

## Step 2: Import Libraries & Configure Setup
We import our libraries and define the base URL structure, as well as headers to mimic a normal browser.

In [ ]:
import os
import time
import random
import requests
from bs4 import BeautifulSoup

# Base URL for the novel chapters
BASE_URL = "https://novellunar.com/novel/pick-me-up-infinite-gacha/chapter/{}"

# Headers to prevent simple bot blocking
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://novellunar.com/novel/pick-me-up-infinite-gacha"
}

## Step 3: Define Extraction and Helper Functions
These helper functions handle:
1. Formatting/cleaning the text (removing redundant empty lines).
2. Fetching and parsing the HTML for a single chapter.

In [ ]:
def clean_text(text):
    """
    Cleans up redundant whitespaces and consecutive empty lines.
    """
    lines = [line.strip() for line in text.split('\n')]
    cleaned_lines = []
    prev_was_empty = False
    for line in lines:
        if not line:
            if not prev_was_empty:
                cleaned_lines.append("")
                prev_was_empty = True
        else:
            cleaned_lines.append(line)
            prev_was_empty = False
    return "\n".join(cleaned_lines)

def fetch_chapter(chapter_num, session):
    """
    Fetches a single chapter and returns its title and parsed content.
    """
    url = BASE_URL.format(chapter_num)
    try:
        response = session.get(url, headers=HEADERS, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Extract chapter title
            title_tag = soup.find('h1')
            title = title_tag.get_text(strip=True) if title_tag else f"Chapter {chapter_num}"
            
            # Extract article content
            article = soup.find('article')
            if not article:
                print(f"Warning: No article found for chapter {chapter_num}")
                return None, None
            
            content_div = article.find('div')
            if not content_div:
                print(f"Warning: No content div found for chapter {chapter_num}")
                return None, None
            
            raw_text = content_div.get_text()
            cleaned_content = clean_text(raw_text)
            return title, cleaned_content
        else:
            print(f"Failed to load chapter {chapter_num} - Status Code: {response.status_code}")
            return None, None
    except Exception as e:
        print(f"Error fetching chapter {chapter_num}: {e}")
        return None, None

## Step 4: Test Fetching a Single Chapter
Let's test our scraper on Chapter 1 to verify everything is working perfectly and see how the parsed text looks before running a bulk download.

In [ ]:
test_session = requests.Session()
title, content = fetch_chapter(1, test_session)

if title and content:
    print(f"=== TEST SUCCESS ===\nTitle: {title}\n")
    print("Preview of content (first 500 chars):")
    print(content[:500] + "...")
else:
    print("Test failed. Please verify your internet connection or the website URL.")

## Step 5: Run the Bulk Downloader
This cell will loop through chapters 1 to 400, fetching each page, parsing it, and saving it into a local `chapters` directory. 

> **Important:** We've added a random delay (`time.sleep`) of 1.0 to 2.5 seconds between requests to be polite to the host server and avoid getting IP banned.

In [ ]:
start_chapter = 1
end_chapter = 400
output_dir = "chapters"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

session = requests.Session()
print(f"Downloading chapters {start_chapter} to {end_chapter}...")

for ch in range(start_chapter, end_chapter + 1):
    title, content = fetch_chapter(ch, session)
    
    if title and content:
        # Clean the title for safe filenames
        safe_title = "".join([c for c in title if c.isalpha() or c.isdigit() or c==' ' or c=='-']).rstrip()
        filename = f"{ch:03d}_{safe_title.replace(' ', '_')}.md"
        filepath = os.path.join(output_dir, filename)
        
        # Format as Markdown
        markdown_content = f"# {title}\n\n{content}\n"
        
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(markdown_content)
        print(f"Saved Chapter {ch}: {filename}")
    else:
        print(f"Skipped Chapter {ch} due to an error.")
        
    # Respectful delay
    delay = random.uniform(1.0, 2.5)
    time.sleep(delay)

print("Finished downloading all chapters!")